# Revision Experiments — internal-review response pack

New experiments answering the 2026-07-16 internal review (see `REVISION_PLAN.md`).
Run **SETUP** first on every fresh runtime, then the cells in order. Every cell
reuses Drive caches from E0–E8 and skips itself if already done.

| Cell | Review issue it closes | Runtime | Wall-clock |
|------|------------------------|---------|------------|
| E4b | Critical: "0.5 defect threshold is arbitrary" — cal-selected per-label thresholds + sensitivity sweep | CPU | ~2 min |
| E5c | Critical: "gated ARR can be misinterpreted" — four explicit gated quantities with raw counts + CIs | CPU | ~5 min (bootstrap) |
| E5d | Critical: "no trivial recovery baselines" — always-framing / prevalence / random / oracle policies | CPU | ~2 min |
| E10 | Critical: "answerability head does not use the question" — question-conditioned triage battery (9 variants) | **GPU (any)** | ~30–60 min (CLIP text extraction + 6×5 head trainings) |
| E7e | "Oracle interpretation is overstated" — thresholded/PCA/permuted-target controls | GPU (any; small MLPs) | ~20 min |
| E7f | "Baselines that should be added" — answer-length & question-length selective baselines | CPU | ~5 min |
| E8f | Major: "internal result values disagree" — locked manifest + `paper_numbers.tex` macros + split-accuracy table | CPU | ~2 min |
| E6c | Major: "VQA models too limited" — modern VLM (BLIP-2) third answerer harvest. **Optional, most expensive.** | **GPU (A100)** | ~1–2 h |

**Cheapest workflow:** one GPU session (any tier): SETUP → E4b → E5c → E5d → E10 → E7e → E7f → E8f.
Run E6c later on an A100 when you want the third-answerer robustness table.

After a run, paste the printed summaries back into the chat / commit the
`results/` JSONs — the manuscript TODO markers map 1:1 to these outputs.

## SETUP — clone repo + minimal deps *(same cell as the master notebook)*

In [ ]:
# ====================================================================
#  SETUP - run FIRST on every fresh runtime (any runtime type).
#  Clones/updates the repo and installs ONLY missing packages.
#  Never reinstalls or downgrades anything Colab already ships
#  (that is what used to break the NumPy/pandas binary stack).
#  Re-running on a warm runtime finishes in seconds.
# ====================================================================
import importlib.util, os, subprocess, sys

REPO_URL  = 'https://github.com/meteorboyF/VQA-paper.git'
REPO_ROOT = '/content/VQA-paper'

# ── Optional switches (set BEFORE anything imports src) ──────────────
# os.environ['VQA_FORCE_RERUN'] = '1'                      # ignore all caches/DONE markers
# os.environ['VQA_BACKBONES']   = 'clip,mobilenet,dinov2'  # full 3-backbone table (A100 recommended)
# os.environ['VQA_DRIVE_BASE']  = '/content/drive/MyDrive/VQA_ML/AVA_VizWiz'  # if your Drive layout differs

def sh(args, check=False):
    print('$', ' '.join(args))
    proc = subprocess.run(args, text=True)
    if check and proc.returncode != 0:
        raise RuntimeError(f'command failed ({proc.returncode}): {args}')
    return proc

# 1) Clone or update to the latest push on main
if not os.path.exists(REPO_ROOT):
    sh(['git', 'clone', '--depth', '1', REPO_URL, REPO_ROOT], check=True)
else:
    if sh(['git', '-C', REPO_ROOT, 'pull', '--ff-only']).returncode != 0:
        print('[setup] pull failed; hard-resetting the (disposable) clone to origin/main')
        sh(['git', '-C', REPO_ROOT, 'fetch', 'origin', 'main'], check=True)
        sh(['git', '-C', REPO_ROOT, 'reset', '--hard', 'origin/main'], check=True)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)
head = subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip()
print(f'[setup] repo at {REPO_ROOT}, HEAD={head}')

# 2) Install ONLY what is missing (import-probe first, zero pip on warm runtimes)
NEEDED = {  # import name -> pip spec
    'torch': 'torch', 'torchvision': 'torchvision',
    'numpy': 'numpy', 'pandas': 'pandas', 'pyarrow': 'pyarrow',
    'sklearn': 'scikit-learn', 'scipy': 'scipy',
    'matplotlib': 'matplotlib', 'PIL': 'Pillow', 'tqdm': 'tqdm',
    'transformers': 'transformers>=4.44',
    'open_clip': 'open-clip-torch>=2.26',
    'timm': 'timm>=1.0',
    'einops': 'einops', 'ftfy': 'ftfy', 'regex': 'regex',
}
missing = [spec for mod, spec in NEEDED.items()
           if importlib.util.find_spec(mod) is None]
if missing:
    print('[setup] installing missing packages:', missing)
    sh([sys.executable, '-m', 'pip', 'install', '-q', '--progress-bar', 'off',
        *missing], check=True)
else:
    print('[setup] all packages already present - no pip work needed.')

# 3) Safety net: verify the compiled numeric stack imports; repair only if broken
r = sh([sys.executable, 'scripts/colab_preflight.py'])
if r.returncode == 10:
    raise SystemExit('Numeric stack was repaired. Runtime -> Restart runtime, '
                     'then rerun this SETUP cell before continuing.')
if r.returncode != 0:
    raise RuntimeError('Numeric stack check failed - see output above.')

print('[setup] READY. Run the next cells - finished experiments skip themselves.')


In [ ]:
# ====================================================================
#  E4b - Guidance-threshold selection & sensitivity
#  RUNTIME: CPU High-RAM.
#  Skips itself instantly if already completed (DONE marker on Drive).
# ====================================================================
import os, sys
REPO_ROOT = '/content/VQA-paper'
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)
from src.experiments import e4b_thresholds
e4b_thresholds.main()

In [ ]:
# ====================================================================
#  E5c - Explicit gated-metric denominators & counts
#  RUNTIME: CPU High-RAM.
#  Skips itself instantly if already completed (DONE marker on Drive).
# ====================================================================
import os, sys
REPO_ROOT = '/content/VQA-paper'
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)
from src.experiments import e5c_explicit_denominators
e5c_explicit_denominators.main()

In [ ]:
# ====================================================================
#  E5d - Trivial guidance-policy baselines
#  RUNTIME: CPU High-RAM.
#  Skips itself instantly if already completed (DONE marker on Drive).
# ====================================================================
import os, sys
REPO_ROOT = '/content/VQA-paper'
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)
from src.experiments import e5d_guidance_baselines
e5d_guidance_baselines.main()

In [ ]:
# ====================================================================
#  E10 - Question-conditioned triage (Critical #1)
#  RUNTIME: GPU (any tier).
#  Skips itself instantly if already completed (DONE marker on Drive).
# ====================================================================
import os, sys
REPO_ROOT = '/content/VQA-paper'
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)
from src.experiments import e10_question_triage
e10_question_triage.main()

In [ ]:
# ====================================================================
#  E7e - Oracle-interpretation controls
#  RUNTIME: GPU (any tier; trains small MLPs).
#  Skips itself instantly if already completed (DONE marker on Drive).
# ====================================================================
import os, sys
REPO_ROOT = '/content/VQA-paper'
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)
from src.experiments import e7e_oracle_controls
e7e_oracle_controls.main()

In [ ]:
# ====================================================================
#  E7f - Shallow selective-prediction baselines
#  RUNTIME: CPU High-RAM.
#  Skips itself instantly if already completed (DONE marker on Drive).
# ====================================================================
import os, sys
REPO_ROOT = '/content/VQA-paper'
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)
from src.experiments import e7f_selective_baselines
e7f_selective_baselines.main()

In [ ]:
# ====================================================================
#  E8f - Locked results manifest + paper_numbers.tex
#  RUNTIME: CPU High-RAM.
#  Skips itself instantly if already completed (DONE marker on Drive).
# ====================================================================
import os, sys
REPO_ROOT = '/content/VQA-paper'
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)
from src.experiments import e8f_paper_numbers
e8f_paper_numbers.main()

## Optional: E6c — modern VLM third answerer (A100 recommended)

Set `VQA_MODEL_ID_3` in the SETUP cell to override the default `Salesforce/blip2-opt-2.7b`.

In [ ]:
# ====================================================================
#  E6c - Modern VLM confidence harvest (OPTIONAL)
#  RUNTIME: GPU - A100 recommended.
#  Skips itself instantly if already completed (DONE marker on Drive).
# ====================================================================
import os, sys
REPO_ROOT = '/content/VQA-paper'
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)
from src.experiments import e6c_vqaconf_vlm
e6c_vqaconf_vlm.main()